# Indian Railways Train Delay Prediction - Full Pipeline

This notebook runs the complete ML pipeline from data loading to submission generation.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_utils import load_data, preprocess_data, create_train_val_split
from features import engineer_features
from models import train_all_models, evaluate_model, get_top_features
from submission import generate_submission, generate_ensemble_submission, save_model

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries loaded successfully!")

## 1. Load Data

In [ ]:
# Load datasets
train, test = load_data('../data/')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain columns: {list(train.columns)}")

In [ ]:
# Quick data overview
print("Target distribution:")
print(train['is_delayed'].value_counts(normalize=True))
print(f"\nDelay rate: {train['is_delayed'].mean():.2%}")

## 2. Preprocess Data

In [ ]:
# Preprocess
train_processed, test_processed = preprocess_data(train, test)

print(f"Processed train shape: {train_processed.shape}")
print(f"Processed test shape: {test_processed.shape}")

## 3. Feature Engineering

In [ ]:
# Apply feature engineering
train_fe, test_fe = engineer_features(train_processed, test_processed)

print(f"Feature engineered train shape: {train_fe.shape}")
print(f"Feature engineered test shape: {test_fe.shape}")
print(f"\nNew features: {[c for c in train_fe.columns if c not in train_processed.columns]}")

## 4. Train/Validation Split

In [ ]:
# Prepare features
X = train_fe.drop('is_delayed', axis=1)
y = train_fe['is_delayed']

# Store journey_ids for test
test_journey_ids = test_fe['journey_id']

# Remove journey_id from features
X = X.drop('journey_id', axis=1)
test_features = test_fe.drop('journey_id', axis=1)

# Split
X_train, X_val, y_train, y_val = create_train_val_split(train_fe, test_size=0.2, random_state=42)
X_train = X_train.drop('journey_id', axis=1)
X_val = X_val.drop('journey_id', axis=1)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_train mean: {y_train.mean():.4f}")
print(f"y_val mean: {y_val.mean():.4f}")

## 5. Train Models

In [ ]:
# Train all models
results = train_all_models(X_train, X_val, y_train, y_val)

In [ ]:
# Print AUC scores
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
for name, result in results.items():
    print(f"{name:20s}: AUC = {result['auc']:.4f}")

In [ ]:
# Detailed evaluation
for name, result in results.items():
    evaluate_model(y_val, result['predictions'], name)

## 6. Feature Importance

In [ ]:
# Plot feature importance for best model
best_model_name = max(results, key=lambda x: results[x]['auc'])
print(f"Best model: {best_model_name}")

if 'feature_importance' in results[best_model_name]:
    top_features = get_top_features(results[best_model_name], n=20)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(data=top_features, y='feature', x='importance')
    plt.title(f'Top 20 Features - {best_model_name}')
    plt.tight_layout()
    plt.show()

## 7. Generate Submissions

In [ ]:
# Generate submissions for all models
for name, result in results.items():
    generate_submission(result['model'], test_features, test_journey_ids, name, '../submissions/')

In [ ]:
# Create weighted ensemble
# You can adjust weights based on validation AUC
weights = {
    'xgboost': 0.35,
    'lightgbm': 0.35,
    'random_forest': 0.15,
    'gradient_boosting': 0.15
}

ensemble_file = generate_ensemble_submission(
    results, test_features, test_journey_ids, 
    weights=weights, output_dir='../submissions/'
)

## 8. Save Models

In [ ]:
# Save best model
save_model(results[best_model_name]['model'], f'best_model_{best_model_name}', '../models/')

# Save all models
for name, result in results.items():
    save_model(result['model'], name, '../models/')

## Summary

All done! Check the following:

1. **Models trained:** All 5 models with validation AUC scores
2. **Submissions:** Individual model predictions in `../submissions/`
3. **Ensemble:** Weighted ensemble submission (usually performs best)
4. **Saved models:** Trained models in `../models/` for future use

Submit `submission_ensemble.csv` to Kaggle to see your leaderboard score!